In [12]:
# importing libraries required in this notebook
import logging
from typing import List, Set, Dict, Any, AnyStr, Optional
from pydantic import BaseModel, AnyUrl, EmailStr, ValidationError

In [19]:
from pydantic import BaseModel

#1 blueprint of class 
class Student(BaseModel):
    name: str 
    age: int
    gpa: float

#2 feeding data 
assign_data = {
    'name' : 'majid',
    'age' :   22,
    'gpa' : '3.4'
}

#3 Object of class
std = Student(**assign_data)

print(f"Name: {std.name}")
print(f"age: {std.age}")
print(f"gpa: {std.gpa} with type: {type(std.gpa)}")    #if I pass this in simpe python it would give me string not float that is the power of pydantic validation

# I can access any attributes wheter it json or dictionary directly through the object of class and attribute name

Name: majid
age: 22
gpa: 3.4 with type: <class 'float'>


In [ ]:
assign_data = {
    'name' : 'majid',
    'age' :   22,
    'gpa' : '3.4'
}

print(f"Type of gpa: {type(assign_data['gpa'])}")   # We can check here in simple python

Type of gpa: <class 'str'>


In [ ]:
# What happened when i pass wrong data types ( age instead of int to completely string in twenty two)


from pydantic import BaseModel

#1 blueprint of class 
class Student(BaseModel):
    name: str 
    age: int
    gpa: float

#2 feeding data 
assign_data = {
    'name' : 'majid',
    'age' : '22',     #this is coerce in pydantic 
    'age1' : 'twenty two',    #not coerce 
    'gpa' : '3.4'
}

#3 Object of class
std = Student(**assign_data)

std.name

'majid'

In [18]:
# how to handle error 
from pydantic import BaseModel, ValidationError

#1 blueprint of class 
class Student(BaseModel):
    name: str 
    age: int
    gpa: float

#2 feeding data 
assign_data = {
    'name' : 'majid',
    'age' : 'twenty two',    #not coerce 
    'gpa' : '3.4'
}

#3 Object of class
try:
    std = Student(**assign_data)
    print(f"Name of Student is: {std.name} with type: {type(std.name)}")
except ValidationError as e:
    for error in e.errors():
        print(f"Error occured: {error['msg']}")
        print(f"Error occured: {error['loc'][0]}")

Error occured: Input should be a valid integer, unable to parse string as an integer
Error occured: age


In [33]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('VectorParsing')

# custome exception handling
class VectorMetadataError(Exception):
    pass

class VectorMetadata(BaseModel):
    vector_id: int
    embedding_dim: int
    is_normalized: bool

def process_vector_dim(raw_payload: dict[str , Any]) -> VectorMetadata:
    try:
        parse_vector = VectorMetadata(**raw_payload)
        logger.info("Vector is successfully ingested: %s",parse_vector.embedding_dim)
        logger.info(f"Type of is_normalized: {type(parse_vector.is_normalized)}")
        return parse_vector
    except ValidationError as error:
        logger.error("Vector is failed to parse: %s" ,raw_payload)
        for err in error.errors():
            raise VectorMetadataError(f"Error: {err['loc'][0]}") from error

if __name__ == '__main__':
    raw_vector = {
    "vector_id": "550",
    "embedding_dim": "1536",
    "is_normalized": "1"  # "1" should coerce to True in Pydantic
}
    result = process_vector_dim(raw_vector)
    print(result)



INFO:VectorParsing:Vector is successfully ingested: 1536
INFO:VectorParsing:Type of is_normalized: <class 'bool'>


vector_id=550 embedding_dim=1536 is_normalized=True


In [43]:
# Task2: checking transaction payment with strict model 
from pydantic import ConfigDict

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("TransactionTrace")

class PaymentTransactError(Exception):
    def __init__(self, message, error):
        super().__init__(message)
        self.error = error

class FinancialTransaction(BaseModel):
    model_config = ConfigDict(strict=True)
    transaction_id: int
    amount: float
    is_flagged: bool

def process_amount(recipte: dict[str, Any]) -> FinancialTransaction:
    try:
        reciption = FinancialTransaction(**recipte)
        logger.info(f"Amount is Successfully transfered with ID: {reciption.transaction_id}")
        return reciption
    except ValidationError as e:
        logger.error(f"Transaction failed for payload: {recipte}")
        raise PaymentTransactError(
            message="Invalid payments payload types",
            error = e.errors()
        ) from e


if __name__ == "__main__":
    recipte = {
        'transaction_id': 222,
        'amount': 43.8,
        'is_flagged': 1
    }
    valid_receipt = {
        'transaction_id': 222,
        'amount': 43.8,
        'is_flagged': False  
    }

    try:
        logger.info(f"User is trying to send money")
        process_amount(recipte)
    except PaymentTransactError as e:
        print(f"Caught Error {e}")
        print(f"Error detail: {e.error}")

    result = process_amount(valid_receipt)
    print("Transaction successfully transferred")
    print("Detail")
    print(result)

INFO:TransactionTrace:User is trying to send money
ERROR:TransactionTrace:Transaction failed for payload: {'transaction_id': 222, 'amount': 43.8, 'is_flagged': 1}
INFO:TransactionTrace:Amount is Successfully transfered with ID: 222


Caught Error Invalid payments payload types
Error detail: [{'type': 'bool_type', 'loc': ('is_flagged',), 'msg': 'Input should be a valid boolean', 'input': 1, 'url': 'https://errors.pydantic.dev/2.13/v/bool_type'}]
Transaction successfully transferred
Detail
transaction_id=222 amount=43.8 is_flagged=False


In [ ]:
invalid_receipt = {
        'transaction_id': 222,
        'amount': 43.8,
        'is_flagged': 1  # Int 1 will fail under strict=True!
    }

    # Case 2: Valid Strict Receipt
valid_receipt = {
        'transaction_id': 222,
        'amount': 43.8,
        'is_flagged': False  # Exact Bool required
    }

In [45]:
import logging
from typing import Any
from pydantic import BaseModel, ConfigDict, ValidationError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("TransactionTrace")

class PaymentTransactError(Exception):
    def __init__(self, message: str, error: list[dict]):
        super().__init__(message)
        self.error = error

class FinancialTransaction(BaseModel):
    model_config = ConfigDict(strict=True)
    transaction_id: int
    amount: float
    is_flagged: bool  # Standardized double 'g'

def process_amount(receipt: dict[str, Any]) -> FinancialTransaction:
    try:
        transaction = FinancialTransaction(**receipt)
        logger.info(f"Amount is Successfully transferred with ID: {transaction.transaction_id}")
        return transaction
    except ValidationError as e:
        logger.error(f"Transaction failed for payload: {receipt}")
        raise PaymentTransactError(
            message="Invalid payments payload types",
            error=e.errors()
        ) from e


if __name__ == "__main__":
    invalid_receipt = {
        'transaction_id': 222,
        'amount': 43.8,
        'is_flagged': 1  # Int 1 fails under strict=True
    }
    
    valid_receipt = {
        'transaction_id': 222,
        'amount': 43.8,
        'is_flagged': False  # Exact Bool
    }

    print("--- Test 1: Invalid Payload ---")
    try:
        logger.info("User is trying to send money (invalid payload)")
        process_amount(invalid_receipt)
    except PaymentTransactError as e:
        print(f"Caught Error: {e}")
        print(f"Error detail: {e.error}\n")

    print("--- Test 2: Valid Payload ---")
    result = process_amount(valid_receipt)
    print(f"Transaction successfully transferred: {result}")

INFO:TransactionTrace:User is trying to send money (invalid payload)
ERROR:TransactionTrace:Transaction failed for payload: {'transaction_id': 222, 'amount': 43.8, 'is_flagged': 1}
INFO:TransactionTrace:Amount is Successfully transferred with ID: 222


--- Test 1: Invalid Payload ---
Caught Error: Invalid payments payload types
Error detail: [{'type': 'bool_type', 'loc': ('is_flagged',), 'msg': 'Input should be a valid boolean', 'input': 1, 'url': 'https://errors.pydantic.dev/2.13/v/bool_type'}]

--- Test 2: Valid Payload ---
Transaction successfully transferred: transaction_id=222 amount=43.8 is_flagged=False


In [52]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('Transaction')

class PaymentError(Exception):
    def __init__(self, message: str, error: list[dict]):
        super().__init__(message)
        self.error = error

class FinancialTransaction(BaseModel):
    model_config = ConfigDict(strict=True)
    transaction_id: int
    amount: float
    is_flagged: bool
def process_transaction(incoming_recipte: dict[str, Any]) -> FinancialTransaction:
    try:
        receiption = FinancialTransaction(**incoming_recipte)
        logger.info(f"Transaction successfully transferred: {receiption.transaction_id}")
        return receiption
    except ValidationError as e:
        logger.error(f'Transaction failed for payload {incoming_recipte}')
        raise PaymentError(
            message = 'Invalid payment payload',
            error = e.errors()

        ) from e

if __name__ == "__main__":
    
    invalid_receipt = {
    'transaction_id': 222,
    'amount': 43.8,
    'is_flagged': 1  # Int 1 will fail under strict=True!
    }

    # Case 2: Valid Strict Receipt
    valid_receipt = {
        'transaction_id': 222,
        'amount': 43.8,
        'is_flagged': False  # Exact Bool required
        }
    try:
        print("Test1: Invalid receiption description")
        logger.info("User is trying to send money ")
        process_transaction(invalid_receipt)
    except PaymentError as e:
        print(f"Caught error {e}")
        print(f"Error Detail: {e.error}")

    print("Test2: Valid Description and recipte")
    result = process_transaction(valid_receipt)
    print(f"Transaction successfully Transferred")
    print(result)

    





INFO:Transaction:User is trying to send money 
ERROR:Transaction:Transaction failed for payload {'transaction_id': 222, 'amount': 43.8, 'is_flagged': 1}
INFO:Transaction:Transaction successfully transferred: 222


Test1: Invalid receiption description
Caught error Invalid payment payload
Error Detail: [{'type': 'bool_type', 'loc': ('is_flagged',), 'msg': 'Input should be a valid boolean', 'input': 1, 'url': 'https://errors.pydantic.dev/2.13/v/bool_type'}]
Test2: Valid Description and recipte
Transaction successfully Transferred
transaction_id=222 amount=43.8 is_flagged=False


In [1]:
# basic configuration 
import logging
from pydantic import BaseModel, AnyUrl, EmailStr, ValidationError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('ScrapTracer')

class ScraperTargetError(Exception):
    def __init__(self, message: str , error: dict[str, Any]):
        super().__init__(message)
        self.error = error


class ScraperTarget(BaseModel):
    target_url: AnyUrl
    admin_email: EmailStr
    max_pages: int|None = 10
    keywords: str | list[str]

def process_scraper_configs(raw_dict: dict[str, Any]) -> ScraperTarget:
    try:
        return ScraperTarget(**raw_dict)
    except ValidationError as err:
        error_summary = []
        for e in err.errors():
            field_path = "->".join(str(loc) for loc in e['loc'])
            error_summary.append(f"[{field_path}] {e['msg']} (Got: {e['input']})")

        raise ScraperTargetError(
            message= "Payload failure accured.",
            error = error_summary
        ) from err


In [3]:
# testing phase of above implemented code: 
if __name__ == "__main__":
        # Test Payload 1: Fully Valid (Keywords as a single string, max_pages as int)
    valid_payload_single = {
        "target_url": "https://news.ycombinator.com",
        "admin_email": "crawler@ai-data.com",
        "max_pages": 5,
        "keywords": "artificial intelligence"
    }

    # Test Payload 2: Fully Valid (Keywords as a list, max_pages as None)
    valid_payload_multi = {
        "target_url": "htt://pythonorg",
        "admin_email": "dev@python.org",
        "max_pages": None,
        "keywords": ["pydantic", "asyncio", "type-hints"]
    }

    # Test Payload 3: Invalid (Invalid email format -> Should raise ScraperConfigError)
    invalid_payload = {
        "target_url": "https://scraping-service.com",
        "admin_email": "majid.12@g.com",  # Invalid Email!
        "max_pages": 20,
        "keywords": "data science"
    }

    
    print("......Testing Valided Payload.....")
    try:
        process_scraper_configs(valid_payload_single)
        print(f"Successfully scraped data payload: {valid_payload_single}")
    except ScraperTargetError as err:
        for e in err.error:
            print(e)
    

......Testing Valided Payload.....
Successfully scraped data payload: {'target_url': 'https://news.ycombinator.com', 'admin_email': 'crawler@ai-data.com', 'max_pages': 5, 'keywords': 'artificial intelligence'}


In [4]:
print(".....Testing Multi String valid payload.....")
try:
    process_scraper_configs(invalid_payload)
except ScraperTargetError as e:
    print("Caught Formated pipeline error")
    for err in e.error:
        print(err)

.....Testing Multi String valid payload.....
